In [ ]:
from platform import python_version
print(python_version())

## Samtools and bcftools

### About BCFtools

BCFtools is a program for variant calling and manipulating files in the Variant Call Format (VCF) and its binary counterpart BCF. All commands work transparently with both VCFs and BCFs, both uncompressed and BGZF-compressed. In order to avoid tedious repetition, throughout this document we will use "VCF" and "BCF" interchangeably, unless specifically noted.

Most commands accept VCF, bgzipped VCF and BCF with filetype detected automatically even when streaming from a pipe. Indexed VCF and BCF work in all situations. Unindexed VCF and BCF and streams work in most, but not all situations. In general, whenever multiple VCFs are read simultaneously, they must be indexed and therefore also compressed.

https://samtools.github.io/bcftools/howtos/index.html

https://github.com/samtools/bcftools

### About GATK

[GATK](https://gatk.broadinstitute.org/hc/en-us)

### About Mustect2

[Mutect2](https://gatk.broadinstitute.org/hc/en-us/articles/360037593851-Mutect2)

In [5]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/mutation_analysis")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir, pdreadcsv, pdwritecsv

root_figure = create_dir(ROOT0, "pictures")
root_results = create_dir(ROOT0, "results")
root_data = create_dir(ROOT0, "data")
root_vcf = create_dir(root_data, "vcf")

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/mutation_analysis
ROOT_SRC added: /home/flavio/uv/mutation_analysis/src


I'll put the design and what to measure in this reply, and write the commands into three files: a runnable-but-unexecuted pipeline script, a sample sheet with the FASTQ layout, and a small Python helper for the clonality input. Nothing will be downloaded or run.Everything is in the three files. `bash -n` and `py_compile` pass. The script only prints a notice unless it is run with `--run`, so it downloads nothing and runs nothing.

### a) Pipeline design

I've assumed these are WGS reads (FASTQ from the three tissues, CRAM from GTEx).

```
FASTQ (T, B, N) ─┐
GTEx CRAM → FASTQ┴▶ BWA-MEM2 → MarkDuplicates → BQSR → QC (depth, contamination, somalier identity)
                                                    │
        ┌───────────────────────┬───────────────────┼──────────────────────┬─────────────────┐
   Germline (N+GTEx)      PoN (GTEx only)     Somatic, per patient     SV / CNV          TEs, mtDNA
   HaplotypeCaller GVCF   Mutect2 → PoN       Mutect2 joint T+B vs N   Manta, Delly      MELT, xTea
   → joint genotyping                         + Strelka2 (2nd caller)  → SURVIVOR         Mutect2 chrM
                                              → FilterMutectCalls      ASCAT purity/CN
                                                    │
                                     VEP → PyClone-VI (T+B CCF) → signatures, MSI
```

#### **What each sample does in the design:**

| Sample | Role | Watch out for |
|---|---|---|
| **N (non-tumour)** | Matched germline control for somatic calling | Tumour cells or field-effect clones in N. If N carries tumour mutations, Mutect2 rejects them as germline. Estimate this with **TINC**. If it is high, use blood as the normal. |
| **B (border)** | Tells you whether the invasion front, field cancerization or pre-malignant clones are present | Low purity. Sequence deeper and expect ASCAT to fail; the helper script has `--purity` to override it. |
| **T (tumour)** | Primary somatic profile | Purity, subclonality, and whole-genome doubling |
| **GTEx** | Panel of normals, plus population germline background | **GTEx WGS comes from blood, not from the tissue**, and from a different sequencing batch. It is not a matched normal and it can't show tissue-level mosaicism. Access requires dbGaP (phs000424). |

#### **Key design choices:**
- **Realign everything from FASTQ** with the same aligner version, including GTEx. Otherwise batch artefacts show up as "somatic" variants and in the panel of normals.
- **Joint multi-sample Mutect2 (T + B, normal N).** This gives read counts for every variant in both T and B, even where one sample has zero alt reads. Clonality needs exactly that.
- **Build the PoN from GTEx, not from N.** Adjacent normal may contain tumour hotspot mutations, which would then be filtered from every patient.
- **Depth:** T ≥ 60–80×, B ≥ 80–100×, N ≥ 30–40×. Low-VAF signal in B and N needs the extra depth.

#### b) What to measure

Ordered by how much each tells you with this design:

1. **Somatic SNVs and indels.** Drivers, TMB, and above all *sharing*: each mutation classified as T-only, T+B, or T+B+N.
2. **Copy number, purity, ploidy, LOH, whole-genome doubling** (ASCAT). Clonality needs these, and they matter on their own.
3. **Clonal architecture ("clones").** Cancer cell fraction per mutation in T and B, clusters, and a phylogeny (e.g., CONIPHER or PairTree after PyClone-VI). This is the central question: does B carry the truncal tumour clone (infiltration), only earlier clones (field cancerization), or its own independent clones?
4. **Structural variants.** Deletions, amplifications, translocations, chromothripsis, fusions. The script uses a consensus of at least 2 callers.
5. **Mutational signatures.** SBS, ID and CN signatures per sample. HRD can be scored with HRDetect or CHORD on top of these. Comparing signatures between T and B shows which processes act early versus late.
6. **MSI** (msisensor-pro).
7. **Germline predisposition** in N, e.g. HR and MMR genes, checked against GTEx and gnomAD frequencies.
8. **Transposable elements.** Somatic L1, Alu and SVA insertions (xTea) are common in some epithelial tumours and can mark clonal lineages. Germline MEIs (MELT) are mostly context. Lower priority.
9. **mtDNA heteroplasmy and copy number.** Cheap lineage markers shared between T and B.
10. **Optional extras from the same BAMs:** telomere length (TelomereHunter) and viral integration (VIRUSBreakend), if the tumour type makes it relevant.

WGS doesn't give you expression, methylation or the immune microenvironment. For those you'd need RNA-seq or methylation data from the same regions.

#### c) Files

- **samplesheet.csv** has one row per lane, with columns `patient, sex, status, sample, lane, fastq_1, fastq_2`. Samples follow the naming `<PT>_T/_B/_N` and `GTEX-*`. It uses the same format as nf-core/sarek.
- **pipeline.sh** has one function per step (0–13) and a `main` that loops over the sample sheet. The header also gives an nf-core/sarek command that covers steps 1–10 if you'd rather use a managed workflow.
- **make_pyclone_input.py** combines the joint Mutect2 VCF with ASCAT segments and purity into PyClone-VI input (cyvcf2 + pandas).

Four things you'll still need to adapt before running:
- The xTea call is left as a comment, because its flags change between releases.
- Germline VQSR or hard filtering after `GenotypeGVCFs` is noted but not written out.
- Strelka2 output isn't intersected with Mutect2 yet. A consensus with a "2 of 2 callers" rule is the usual approach.
- The `ref/` resource paths assume the Broad GRCh38 bundle.

In [8]:
[x for x in os.listdir(root_data) ]

['samplesheet_prostate.csv', 'clinical_prostate.csv', 'vcf', 'samplesheet.csv']

In [7]:
fname = 'samplesheet.csv'

df = pdreadcsv(fname, root_data, sep=',')
df

,patient,sex,status,sample,lane,fastq_1,fastq_2
0,P01,XX,1,P01_T,L001,fastq/P01_T_S1_L001_R1_001.fastq.gz,fastq/P01_T_S1_L001_R2_001.fastq.gz
1,P01,XX,1,P01_T,L002,fastq/P01_T_S1_L002_R1_001.fastq.gz,fastq/P01_T_S1_L002_R2_001.fastq.gz
2,P01,XX,1,P01_B,L001,fastq/P01_B_S2_L001_R1_001.fastq.gz,fastq/P01_B_S2_L001_R2_001.fastq.gz
3,P01,XX,1,P01_B,L002,fastq/P01_B_S2_L002_R1_001.fastq.gz,fastq/P01_B_S2_L002_R2_001.fastq.gz
4,P01,XX,0,P01_N,L001,fastq/P01_N_S3_L001_R1_001.fastq.gz,fastq/P01_N_S3_L001_R2_001.fastq.gz
5,P02,XY,1,P02_T,L001,fastq/P02_T_S4_L001_R1_001.fastq.gz,fastq/P02_T_S4_L001_R2_001.fastq.gz
6,P02,XY,1,P02_B,L001,fastq/P02_B_S5_L001_R1_001.fastq.gz,fastq/P02_B_S5_L001_R2_001.fastq.gz
7,P02,XY,0,P02_N,L001,fastq/P02_N_S6_L001_R1_001.fastq.gz,fastq/P02_N_S6_L001_R2_001.fastq.gz
8,GTEX-DONOR01,XY,0,GTEX-DONOR01,L001,fastq/GTEX-DONOR01_L001_R1.fastq.gz,fastq/GTEX-DONOR01_L001_R2.fastq.gz
9,GTEX-DONOR02,XX,0,GTEX-DONOR02,L001,fastq/GTEX-DONOR02_L001_R1.fastq.gz,fastq/GTEX-DONOR02_L001_R2.fastq.gz


In [10]:
fname = 'samplesheet_prostate.csv'
dfp = pdreadcsv(fname, root_data, sep=',')
dfp

,patient,cohort,group,sample,bam
0,PR01,cancer,T,PR01_T1,ega/PR01_T1.bam
1,PR01,cancer,T,PR01_T2,ega/PR01_T2.bam
2,PR01,cancer,B,PR01_B,ega/PR01_B.bam
3,PR01,cancer,N,PR01_N,ega/PR01_N.bam
4,PR02,cancer,T,PR02_T1,ega/PR02_T1.bam
5,PR02,cancer,B,PR02_B,ega/PR02_B.bam
6,PR02,cancer,N,PR02_N,ega/PR02_N.bam
7,HP01,healthy,H,HP01_H,ega/HP01_H.bam
8,HP01,healthy,N,HP01_N,ega/HP01_N.bam
9,HP02,healthy,H,HP02_H,ega/HP02_H.bam


In [11]:
fname = 'clinical_prostate.csv'
dfc = pdreadcsv(fname, root_data, sep=',')
dfc


,patient,cohort,isup_grade_group,gleason,pT,pN,M
0,PR01,cancer,NaN,NaN,NaN,NaN,NaN
1,PR02,cancer,NaN,NaN,NaN,NaN,NaN
2,HP01,healthy,NaN,NaN,NaN,NaN,NaN
3,HP02,healthy,NaN,NaN,NaN,NaN,NaN
